# 4대 카메라 네트워크 녹화

이 노트북 하나를 **서버 PC와 송신 장비 4대에서 공통으로 사용**합니다.

- pinky_01, pinky_02: Pinky-Pro 내장 카메라
- fixed_01, fixed_02: 일반 PC에 USB로 연결한 HCAM01L
- 서버 PC: 네 스트림을 받아 카메라별 MP4와 프레임 타임스탬프 CSV 저장
- RealSense와 OMX-AI 손목 카메라는 현재 범위에서 제외

처음에는 640×480, 15 FPS, 30초로 시험하고 안정적이면 30 FPS와 원하는 시간으로 올리세요.

## 실행 순서

1. 모든 장비를 같은 유선 LAN에 연결합니다.
2. 서버 PC에서 ROLE을 server로 설정한 뒤 마지막 실행 셀을 실행합니다.
3. Pinky/일반 PC에서 ROLE을 sender로 설정하고 자기 CAMERA_ID를 입력합니다.
4. 송신 장비 4대에서 마지막 실행 셀을 실행합니다.
5. 서버가 네 장비 연결을 모두 확인한 후 공통 시작 신호를 보냅니다.
6. 녹화가 끝나면 서버의 세션 폴더를 확인합니다.

> Pinky 내부 디스크에는 영상을 저장하지 않습니다. 프레임을 메모리에서 압축해 서버로 바로 전송합니다.

## 1. Ubuntu 24.04 준비

서버 PC와 일반 PC:

    sudo apt update
    sudo apt install -y python3-opencv python3-numpy python3-jupyter v4l-utils

Pinky-Pro에는 기존 pinkylib와 OpenCV 환경을 사용합니다.

서버 방화벽을 사용하는 경우:

    sudo ufw allow 5001:5004/tcp

서버 IP 확인:

    hostname -I
    ip -br addr

서버 IP는 고정 IP 또는 공유기의 DHCP 예약을 사용하는 것이 좋습니다.

## 2. 역할·카메라·포트 설정

| 실행 장비 | ROLE | CAMERA_ID | SOURCE_TYPE |
|---|---|---|---|
| 서버 PC | server | 사용 안 함 | 사용 안 함 |
| Pinky-Pro 1 | sender | pinky_01 | pinky |
| Pinky-Pro 2 | sender | pinky_02 | pinky |
| 일반 PC 1 | sender | fixed_01 | opencv |
| 일반 PC 2 | sender | fixed_02 | opencv |

- SERVER_IP: 서버 PC 주소
- CAMERA_DEVICE: 일반 PC의 /dev/video* 또는 /dev/v4l/by-id/*
- RECORD_SECONDS: 서버 값이 네 송신 장비에 전달됩니다.

In [ ]:
from pathlib import Path

ROLE = "server"  # "server" 또는 "sender"
CAMERA_ID = "fixed_01"  # sender에서만 사용
SOURCE_TYPE = "opencv"  # sender에서만 사용: "pinky" 또는 "opencv"

SERVER_IP = "192.168.0.10"  # 실제 서버 LAN IP로 수정
BIND_IP = "0.0.0.0"

PORTS = {
    "pinky_01": 5001,
    "pinky_02": 5002,
    "fixed_01": 5003,
    "fixed_02": 5004,
}
EXPECTED_CAMERAS = list(PORTS)

CAMERA_DEVICE = "/dev/video0"
PINKY_FRAME_IS_RGB = True

WIDTH = 640
HEIGHT = 480
FPS = 15
JPEG_QUALITY = 85
RECORD_SECONDS = 30

BASE_RECORD_DIR = Path("/home/syw/Trihouse/dataset/camera_recordings")

## 3. 일반 PC의 USB 카메라 포트 확인

HCAM01L을 한 대만 연결한 뒤 아래 셀을 실행하세요.

- /dev/video0 숫자는 USB 재연결 후 달라질 수 있습니다.
- 가능하면 /dev/v4l/by-id 아래의 고정 경로를 CAMERA_DEVICE로 사용합니다.
- 지원 포맷에서 640×480과 원하는 FPS를 확인합니다.

In [ ]:
import shutil
import subprocess

if ROLE == "sender" and SOURCE_TYPE == "opencv":
    if shutil.which("v4l2-ctl"):
        subprocess.run(["v4l2-ctl", "--list-devices"], check=False)
        print("\n/dev/v4l/by-id:")
        subprocess.run(["ls", "-l", "/dev/v4l/by-id"], check=False)
        print(f"\n현재 설정 장치의 지원 포맷: {CAMERA_DEVICE}")
        subprocess.run(
            ["v4l2-ctl", "--device", str(CAMERA_DEVICE), "--list-formats-ext"],
            check=False,
        )
    else:
        print("v4l2-ctl이 없습니다: sudo apt install v4l-utils")
else:
    print("이 셀은 일반 PC의 opencv sender에서 실행하세요.")

## 4. 네트워크 메시지 규칙

TCP 연결마다 처음 한 번 카메라 정보(JSON)를 보내고, 이후 촬영 시각과 JPEG를 전송합니다.
JPEG 크기 0은 정상 종료를 뜻합니다. 아래 셀은 카메라를 열지 않는 공통 함수입니다.

In [ ]:
import json
import socket
import struct

_LENGTH = struct.Struct("!I")
_FRAME_HEADER = struct.Struct("!QI")
_DURATION = struct.Struct("!d")
_MAX_HANDSHAKE_BYTES = 64 * 1024
_MAX_JPEG_BYTES = 20 * 1024 * 1024


def validate_port_map(port_map: dict[str, int]) -> None:
    if not port_map:
        raise ValueError("PORTS가 비어 있습니다.")
    ports = list(port_map.values())
    if any(not isinstance(port, int) or not 1 <= port <= 65535 for port in ports):
        raise ValueError("모든 포트는 1~65535 범위의 정수여야 합니다.")
    if len(set(ports)) != len(ports):
        raise ValueError("카메라마다 중복되지 않은 포트를 사용해야 합니다.")


def recv_exact(sock: socket.socket, size: int) -> bytes | None:
    chunks = bytearray()
    while len(chunks) < size:
        chunk = sock.recv(size - len(chunks))
        if not chunk:
            if not chunks:
                return None
            raise ConnectionError("메시지를 받는 중 연결이 종료되었습니다.")
        chunks.extend(chunk)
    return bytes(chunks)


def send_handshake(sock: socket.socket, metadata: dict) -> None:
    payload = json.dumps(metadata, ensure_ascii=False).encode("utf-8")
    if len(payload) > _MAX_HANDSHAKE_BYTES:
        raise ValueError("handshake가 너무 큽니다.")
    sock.sendall(_LENGTH.pack(len(payload)) + payload)


def recv_handshake(sock: socket.socket) -> dict:
    size_bytes = recv_exact(sock, _LENGTH.size)
    if size_bytes is None:
        raise ConnectionError("handshake 전에 연결이 종료되었습니다.")
    (size,) = _LENGTH.unpack(size_bytes)
    if not 0 < size <= _MAX_HANDSHAKE_BYTES:
        raise ValueError(f"잘못된 handshake 크기: {size}")
    payload = recv_exact(sock, size)
    if payload is None:
        raise ConnectionError("handshake 본문 전에 연결이 종료되었습니다.")
    return json.loads(payload.decode("utf-8"))


def send_frame(sock: socket.socket, timestamp_ns: int, jpeg_bytes: bytes) -> None:
    if not 0 < len(jpeg_bytes) <= _MAX_JPEG_BYTES:
        raise ValueError(f"잘못된 JPEG 크기: {len(jpeg_bytes)}")
    sock.sendall(_FRAME_HEADER.pack(timestamp_ns, len(jpeg_bytes)) + jpeg_bytes)


def send_end(sock: socket.socket) -> None:
    sock.sendall(_FRAME_HEADER.pack(0, 0))


def recv_frame(sock: socket.socket) -> tuple[int, bytes] | None:
    header = recv_exact(sock, _FRAME_HEADER.size)
    if header is None:
        return None
    timestamp_ns, jpeg_size = _FRAME_HEADER.unpack(header)
    if jpeg_size == 0:
        return None
    if jpeg_size > _MAX_JPEG_BYTES:
        raise ValueError(f"허용 범위를 넘은 JPEG 크기: {jpeg_size}")
    payload = recv_exact(sock, jpeg_size)
    if payload is None:
        raise ConnectionError("JPEG 본문 전에 연결이 종료되었습니다.")
    return timestamp_ns, payload

## 5. 카메라 어댑터

- OpenCVCameraSource: 일반 PC에 USB로 연결한 HCAM01L
- PinkyCameraSource: Pinky-Pro의 pinkylib.Camera
- Pinky 색상이 잘못 보이면 PINKY_FRAME_IS_RGB 값을 바꿔 비교하세요.

In [ ]:
import time

import cv2
import numpy as np


class OpenCVCameraSource:
    def __init__(self, device, width: int, height: int, fps: int):
        self.device = device
        self.width = width
        self.height = height
        self.fps = fps
        self.cap = None

    def start(self) -> None:
        self.cap = cv2.VideoCapture(self.device, cv2.CAP_V4L2)
        if not self.cap.isOpened():
            raise RuntimeError(f"카메라를 열 수 없습니다: {self.device}")
        self.cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, self.width)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.height)
        self.cap.set(cv2.CAP_PROP_FPS, self.fps)

    def read(self):
        ok, frame = self.cap.read()
        return frame if ok else None

    def close(self) -> None:
        if self.cap is not None:
            self.cap.release()


class PinkyCameraSource:
    def __init__(self, frame_is_rgb: bool):
        self.frame_is_rgb = frame_is_rgb
        self.cam = None

    def start(self) -> None:
        from pinkylib import Camera

        self.cam = Camera()
        self.cam.start()

    def read(self):
        frame = self.cam.get_frame()
        if frame is None:
            return None
        frame = np.asarray(frame)
        if self.frame_is_rgb:
            if frame.ndim == 3 and frame.shape[2] == 4:
                return cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR)
            return cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        return frame

    def close(self) -> None:
        if self.cam is None:
            return
        if hasattr(self.cam, "stop"):
            self.cam.stop()
        if hasattr(self.cam, "close"):
            self.cam.close()


def make_camera_source():
    if SOURCE_TYPE == "opencv":
        return OpenCVCameraSource(CAMERA_DEVICE, WIDTH, HEIGHT, FPS)
    if SOURCE_TYPE == "pinky":
        return PinkyCameraSource(PINKY_FRAME_IS_RGB)
    raise ValueError("SOURCE_TYPE은 'opencv' 또는 'pinky'여야 합니다.")

## 6. 송신기 함수

Pinky-Pro 2대와 일반 PC 2대에서 사용합니다. 서버 연결 후 다른 카메라를 기다리고, 서버가 보낸 녹화 시간만큼 전송합니다.

In [ ]:
def run_sender() -> None:
    validate_port_map(PORTS)
    if CAMERA_ID not in PORTS:
        raise ValueError(f"알 수 없는 CAMERA_ID: {CAMERA_ID}")

    source = make_camera_source()
    sock = None
    sent_frames = 0

    try:
        print(f"[{CAMERA_ID}] 카메라를 여는 중...")
        source.start()

        server_address = (SERVER_IP, PORTS[CAMERA_ID])
        print(f"[{CAMERA_ID}] 서버 연결 중: {server_address}")
        sock = socket.create_connection(server_address, timeout=15)
        sock.settimeout(None)

        send_handshake(
            sock,
            {
                "camera_id": CAMERA_ID,
                "source_type": SOURCE_TYPE,
                "width": WIDTH,
                "height": HEIGHT,
                "fps": FPS,
                "jpeg_quality": JPEG_QUALITY,
            },
        )

        print(f"[{CAMERA_ID}] 다른 카메라와 서버 시작 신호를 기다립니다.")
        duration_bytes = recv_exact(sock, _DURATION.size)
        if duration_bytes is None:
            raise ConnectionError("시작 신호 전에 서버 연결이 종료되었습니다.")
        (record_seconds,) = _DURATION.unpack(duration_bytes)
        print(f"[{CAMERA_ID}] {record_seconds:.1f}초 녹화를 시작합니다.")

        started_at = time.monotonic()
        next_frame_at = started_at
        frame_period = 1.0 / FPS

        while time.monotonic() - started_at < record_seconds:
            frame = source.read()
            if frame is None:
                print(f"[{CAMERA_ID}] 프레임 읽기 실패")
                continue
            if frame.shape[1] != WIDTH or frame.shape[0] != HEIGHT:
                frame = cv2.resize(frame, (WIDTH, HEIGHT))

            ok, encoded = cv2.imencode(
                ".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
            )
            if not ok:
                continue

            send_frame(sock, time.time_ns(), encoded.tobytes())
            sent_frames += 1
            if sent_frames % max(FPS * 5, 1) == 0:
                elapsed = time.monotonic() - started_at
                print(f"[{CAMERA_ID}] {sent_frames} 프레임 / {elapsed:.1f}초")

            next_frame_at += frame_period
            sleep_seconds = next_frame_at - time.monotonic()
            if sleep_seconds > 0:
                time.sleep(sleep_seconds)
            else:
                next_frame_at = time.monotonic()

        send_end(sock)
        print(f"[{CAMERA_ID}] 전송 완료: {sent_frames} 프레임")

    finally:
        source.close()
        if sock is not None:
            sock.close()
        print(f"[{CAMERA_ID}] 카메라와 네트워크 연결 종료")

## 7. 서버 녹화 함수

서버는 네 포트를 모두 열고 연결을 기다립니다. 연결이 끝나면 동시에 시작 신호를 보내고 videos, timestamps, session.json을 저장합니다.

In [ ]:
import csv
import threading
from datetime import datetime


def _make_listener(bind_ip: str, port: int) -> socket.socket:
    listener = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    listener.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    listener.bind((bind_ip, port))
    listener.listen(1)
    return listener


def _record_stream(
    camera_id: str,
    conn: socket.socket,
    metadata: dict,
    video_path: Path,
    timestamp_path: Path,
) -> None:
    writer = None
    received_frames = 0
    fps = float(metadata.get("fps", FPS))

    try:
        with timestamp_path.open("w", newline="", encoding="utf-8") as csv_file:
            csv_writer = csv.writer(csv_file)
            csv_writer.writerow(
                ["frame_index", "sender_timestamp_ns", "server_receive_timestamp_ns"]
            )

            while True:
                packet = recv_frame(conn)
                if packet is None:
                    break
                sender_timestamp_ns, jpeg_bytes = packet
                server_receive_timestamp_ns = time.time_ns()
                encoded = np.frombuffer(jpeg_bytes, dtype=np.uint8)
                frame = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
                if frame is None:
                    print(f"[서버:{camera_id}] JPEG 디코딩 실패")
                    continue

                if writer is None:
                    height, width = frame.shape[:2]
                    writer = cv2.VideoWriter(
                        str(video_path),
                        cv2.VideoWriter_fourcc(*"mp4v"),
                        fps,
                        (width, height),
                    )
                    if not writer.isOpened():
                        raise RuntimeError(f"영상 파일을 열 수 없습니다: {video_path}")

                writer.write(frame)
                csv_writer.writerow(
                    [received_frames, sender_timestamp_ns, server_receive_timestamp_ns]
                )
                received_frames += 1
                if received_frames % max(int(fps * 5), 1) == 0:
                    print(f"[서버:{camera_id}] {received_frames} 프레임 저장")
    except Exception as error:
        print(f"[서버:{camera_id}] 오류: {error}")
    finally:
        if writer is not None:
            writer.release()
        conn.close()
        print(f"[서버:{camera_id}] 종료, 총 {received_frames} 프레임")


def run_server() -> Path:
    validate_port_map(PORTS)
    missing = [camera_id for camera_id in EXPECTED_CAMERAS if camera_id not in PORTS]
    if missing:
        raise ValueError(f"PORTS에 없는 EXPECTED_CAMERAS: {missing}")

    session_id = datetime.now().strftime("session_%Y%m%d_%H%M%S")
    session_dir = BASE_RECORD_DIR / session_id
    video_dir = session_dir / "videos"
    timestamp_dir = session_dir / "timestamps"
    video_dir.mkdir(parents=True, exist_ok=False)
    timestamp_dir.mkdir(parents=True, exist_ok=False)

    listeners = {}
    connections = {}
    metadata_by_camera = {}

    try:
        for camera_id in EXPECTED_CAMERAS:
            port = PORTS[camera_id]
            listeners[camera_id] = _make_listener(BIND_IP, port)
            print(f"[서버] {camera_id} 대기: {BIND_IP}:{port}")

        for expected_camera_id in EXPECTED_CAMERAS:
            conn, address = listeners[expected_camera_id].accept()
            metadata = recv_handshake(conn)
            actual_camera_id = metadata.get("camera_id")
            if actual_camera_id != expected_camera_id:
                conn.close()
                raise ValueError(
                    f"포트 {PORTS[expected_camera_id]}: "
                    f"{expected_camera_id} 대신 {actual_camera_id} 연결"
                )
            connections[actual_camera_id] = conn
            metadata_by_camera[actual_camera_id] = metadata
            print(f"[서버] 연결 완료: {actual_camera_id} <- {address}")

        session_metadata = {
            "session_id": session_id,
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "record_seconds": RECORD_SECONDS,
            "expected_cameras": EXPECTED_CAMERAS,
            "ports": PORTS,
            "cameras": metadata_by_camera,
        }
        (session_dir / "session.json").write_text(
            json.dumps(session_metadata, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

        threads = []
        for camera_id, conn in connections.items():
            thread = threading.Thread(
                target=_record_stream,
                args=(
                    camera_id,
                    conn,
                    metadata_by_camera[camera_id],
                    video_dir / f"{camera_id}.mp4",
                    timestamp_dir / f"{camera_id}.csv",
                ),
                daemon=False,
            )
            thread.start()
            threads.append(thread)

        print("[서버] 모든 카메라 연결 완료. 녹화를 시작합니다.")
        duration_packet = _DURATION.pack(float(RECORD_SECONDS))
        for conn in connections.values():
            conn.sendall(duration_packet)

        for thread in threads:
            thread.join()

        print(f"[서버] 세션 저장 완료: {session_dir}")
        return session_dir
    finally:
        for listener in listeners.values():
            listener.close()
        for conn in connections.values():
            try:
                conn.close()
            except OSError:
                pass

## 8. 실행

서버에서 먼저 실행한 뒤 송신 장비 4대에서 실행합니다. 서버는 네 장비가 모두 연결될 때까지 대기합니다.
중지할 때는 Jupyter의 Interrupt를 사용하세요.

In [ ]:
if ROLE == "server":
    LAST_SESSION_DIR = run_server()
elif ROLE == "sender":
    run_sender()
else:
    raise ValueError("ROLE은 'server' 또는 'sender'여야 합니다.")

## 9. 서버 결과 확인

녹화가 끝난 뒤 서버에서 실행합니다. 영상 4개의 크기, 프레임 수, FPS와 길이를 확인합니다.

In [ ]:
if ROLE == "server" and "LAST_SESSION_DIR" in globals():
    print(f"세션: {LAST_SESSION_DIR}")
    for video_path in sorted((LAST_SESSION_DIR / "videos").glob("*.mp4")):
        size_mb = video_path.stat().st_size / (1024 * 1024)
        cap = cv2.VideoCapture(str(video_path))
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        video_fps = cap.get(cv2.CAP_PROP_FPS)
        duration = frame_count / video_fps if video_fps > 0 else 0
        cap.release()
        print(
            f"{video_path.name}: {size_mb:.2f} MB, "
            f"{frame_count} frames, {video_fps:.2f} FPS, {duration:.1f}s"
        )
else:
    print("서버 녹화를 완료한 뒤 실행하세요.")